In [1]:
import intake
import matplotlib.pyplot as plt
import numpy as np
import netCDF4 as nc
import cartopy.crs as ccrs
import xarray as xr
import cmocean as cm
import glob
import matplotlib.colors as col
import cf_xarray as cf
# need to install opencv-python for this:
import cv2
import warnings
import logging
from scipy.ndimage import distance_transform_edt
import scipy.ndimage as nd
from scipy.interpolate import griddata
from matplotlib.animation import FuncAnimation
import imageio.v3 as iio
import glob
import imageio.v3 as iio
import cmocean.cm as cm

logging.captureWarnings(True)
logging.getLogger('py.warnings').setLevel(logging.ERROR)

from dask.distributed import Client


In [2]:
client = Client(threads_per_worker = 1)

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that i

In [3]:
catalog = intake.cat.access_nri

In [4]:
catalog.search(model='ACCESS-OM2', variable = 'temp', frequency = '1mon')

,model,description,realm,frequency,variable
name,,,,,
01deg_jra55_ryf_ENFull,{ACCESS-OM2},"{0.1° ACCESS-OM2 El Níño run for the simulations performed in Huguenin et al. (2024, GRL)}",{ocean},{1mon},{temp}
01deg_jra55_ryf_LNFull,{ACCESS-OM2},"{0.1° ACCESS-OM2 La Níña run for the simulations performed in Huguenin et al. (2024, GRL)}",{ocean},{1mon},{temp}
01deg_jra55v13_ryf9091_qian_wthmp,{ACCESS-OM2},"{Future perturbations with wind, thermal and meltwater forcing, branching off 01deg_jra55v13_ryf9091, as described in Li et al. 2023, https://www.nature.com/articles/s41586-023-05762-w}",{ocean},{1mon},{temp}
01deg_jra55v13_ryf9091_qian_wthp,{ACCESS-OM2},"{Future perturbation with wind and thermal forcing, branching off 01deg_jra55v13_ryf9091, as described in Li et al. 2023, https://www.nature.com/articles/s41586-023-05762-w}",{ocean},{1mon},{temp}
01deg_jra55v150_iaf_cycle1,{ACCESS-OM2},{Cycle 1 of 0.1 degree ACCESS-OM2 global model configuration with JRA55-do v1.5.0 OMIP2 interannual forcing},{ocean},{1mon},{temp}
025deg_era5_iaf,{ACCESS-OM2},{0.25 degree ACCESS-OM2 global model configuration with ERA5 interannual\nforcing (1980-2021)},{ocean},{1mon},{temp}
025deg_era5_ryf,{ACCESS-OM2},{0.25 degree ACCESS-OM2 global model configuration with ERA5 RYF9091 repeat\nyear forcing (May 1990 to Apr 1991)},{ocean},{1mon},{temp}
025deg_jra55_iaf_era5comparison,{ACCESS-OM2},{0.25 degree ACCESS-OM2 global model configuration with JRA55-do v1.5.0\ninterannual forcing (1980-2019)},{ocean},{1mon},{temp}
025deg_jra55_iaf_omip2_cycle1,{ACCESS-OM2},{Cycle 1/6 of 0.25 degree ACCESS-OM2 physics-only global configuration with JRA55-do v1.4 OMIP2 interannual forcing (1958-2019)},{ocean},{1mon},{temp}


In [5]:
experiment = "01deg_jra55v140_iaf_cycle3"
variable="temp"


In [10]:
data_ic = catalog[experiment].search(variable=variable,frequency='1mon').to_dask()


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'argo' loading failed:
Expecting value: line 1 column 1 (char 0)
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


In [13]:
import intake
catalog = intake.cat.access_nri
experiment = "01deg_jra55v13_ryf9091"
variables = catalog[experiment].unique().variable
print(variables)

['ANGLE', 'ANGLET', 'HTE', 'HTN', 'NCAT', 'TLAT', 'TLON', 'Tsfc_m', 'ULAT', 'ULON', 'aice_m', 'aicen_m', 'alidf_ai_m', 'alidr_ai_m', 'alvdf_ai_m', 'alvdr_ai_m', 'blkmask', 'congel_m', 'divu_m', 'dxt', 'dxu', 'dyt', 'dyu', 'flatn_ai_m', 'fmeltt_ai_m', 'fmelttn_ai_m', 'frazil_m', 'frz_onset_m', 'fsalt_ai_m', 'fsalt_m', 'fswup_m', 'hi_m', 'hs_m', 'mlt_onset_m', 'opening_m', 'shear_m', 'sig1_m', 'sig2_m', 'sss_m', 'sst_m', 'strairx_m', 'strairy_m', 'strength_m', 'tarea', 'time', 'time_bounds', 'tmask', 'uarea', 'uatm_m', 'uocn_m', 'uvel_m', 'vatm_m', 'vicen_m', 'vocn_m', 'vvel_m', 'age_global', 'average_DT', 'average_T1', 'average_T2', 'dzt', 'grid_xt_ocean', 'grid_xu_ocean', 'grid_yt_ocean', 'grid_yu_ocean', 'neutral', 'neutralrho_edges', 'nv', 'pot_rho_0', 'pot_temp', 'potrho', 'potrho_edges', 'rho', 'salt', 'st_edges_ocean', 'st_ocean', 'sw_edges_ocean', 'sw_ocean', 'temp', 'temp_xflux_adv', 'temp_yflux_adv', 'tx_trans', 'tx_trans_rho', 'ty_trans', 'ty_trans_nrho_submeso', 'ty_trans_rho

In [22]:

catalog[experiment].search(variable=variable, frequency = '1mon')


,unique
filename,1
path,920
file_id,1
frequency,1
start_date,920
end_date,920
variable,46
variable_long_name,42
variable_standard_name,11
variable_cell_methods,2
